# Euronext-SimStock — pipeline clean

Notebook minimal. Pour les runs longs, préfère `tmux` + CLI. Ce notebook sert surtout à inspecter les objets.

In [ ]:
from pathlib import Path
import sys, logging

# Résoudre la racine du repo : quand le notebook est dans notebooks/,
# .parent remonte au repo root qui contient euronext_simstock/
_nb_dir = Path.cwd()
PROJECT_PATH = _nb_dir.parent if (_nb_dir.parent / 'euronext_simstock').exists() else _nb_dir

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%H:%M:%S')
print('Project root:', PROJECT_PATH)


## Configuration

In [ ]:
from euronext_simstock.main import make_args, run_pipeline

args = make_args(
    sample=True,                 # False + user_csv=Path('mon_univers.csv') pour la prod
    user_csv=None,
    force_refresh=False,
    epochs_per_domain=1,          # augmente à 5 pour un vrai run
    batch_size=256,
    mode='cosine',                # 'dtw' pour la méthode la plus proche de l'expérience
    feature_mode='paper',
    save_name='notebook_smoke_test',
    log_level='INFO',
)
args


## Lancer le pipeline

In [ ]:
engine = run_pipeline(args)
print(f'Engine prêt: {len(engine.tickers)} tickers')


## Inspecter les substitutions

In [ ]:
ticker = engine.tickers[0]
print('Ticker:', ticker)
print(engine.similarity_diagnostics().to_string(index=False))
print('
Substituts:')
for sub, sim in engine.all_substitutes(ticker, threshold=engine.cfg.similarity_threshold)[:20]:
    print(f'{sub:12s} sim={sim:.4f}')


## Comparer deux décisions / portefeuilles

In [ ]:
if len(engine.tickers) >= 4:
    trader_x = {engine.tickers[0]: 100.0, engine.tickers[1]: 50.0}
    trader_y = {engine.tickers[0]: 100.0, engine.tickers[2]: 50.0}
    engine.compare_trades(trader_x, trader_y, threshold=engine.cfg.similarity_threshold)
